# 可选实验：特征缩放与学习率（多变量）

## 目标
在本实验中你将：
- 使用上一个实验开发的多变量例程
- 在含多个特征的数据集上运行梯度下降
- 探讨*学习率 alpha* 对梯度下降的影响
- 通过 z 分数归一化的*特征缩放*改进梯度下降的表现

## 工具
你将使用上一个实验开发的函数，以及 matplotlib 和 NumPy。 

In [ ]:
import numpy as np
np.set_printoptions(precision=2)
import matplotlib.pyplot as plt
dlblue = '#0096ff'; dlorange = '#FF9300'; dldarkred='#C00000'; dlmagenta='#FF40FF'; dlpurple='#7030A0'; 
plt.style.use('./deeplearning.mplstyle')
from lab_utils_multi import  load_house_data, compute_cost, run_gradient_descent 
from lab_utils_multi import  norm_plot, plt_contour_multi, plt_equal_scale, plot_cost_i_w

## 记号

| 一般 记号 | 说明 | Python（如适用） |
|:---|:---|:---|
| $a$ | 标量，非粗体 |  |
| $\mathbf{a}$ | 向量，粗体 |  |
| $\mathbf{A}$ | 矩阵，粗体大写 |  |
| **回归** |  |  |
| $\mathbf{X}$ | 训练样本矩阵 | `X_train` |
| $\mathbf{y}$ | 训练样本目标值 | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `X[i]`, `y[i]` |
| m | 训练样本的数量 | `m` |
| n | 每个样本中的特征数量 | `n` |
| $\mathbf{w}$ | 参数：权重 | `w` |
| $b$ | 参数：偏置 | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 以 $\mathbf{w},b$ 为参数、在 $\mathbf{x}^{(i)}$ 处求得的模型评估结果：$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |
| $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ | 代价关于参数 $w_j$ 的梯度或偏导数 | `dj_dw[j]` |
| $\frac{\partial J(\mathbf{w},b)}{\partial b}$ | 代价关于参数 $b$ 的梯度或偏导数 | `dj_db` |

#  问题陈述

与之前的实验一样，你将使用房价预测这个引入性例子。训练数据集包含许多样本、4 个特征（面积、卧室数、楼层数和房龄），如下表所示。注意，本实验中面积特征的单位是平方英尺，而此前的实验使用 1000 平方英尺。这个数据集比上一个实验更大。

我们希望用这些数值建立一个线性回归模型，从而可以预测其他住房的价格——比如一套 1200 平方英尺、3 间卧室、1 层、房龄 40 年的住房。

##  数据集：
| 面积（平方英尺） | 卧室数  | 楼层数 | 房龄 | 价格（千美元）  |   
| ----------------| ------------------- |----------------- |--------------|----------------------- |  
| 952             | 2                   | 1                | 65           | 271.5                  |  
| 1244            | 3                   | 2                | 64           | 232                    |  
| 1947            | 3                   | 2                | 17           | 509.8                  |  
| ...             | ...                 | ...              | ...          | ...                    |


In [ ]:
# 加载数据集
X_train, y_train = load_house_data()
X_features = ['size(sqft)','bedrooms','floors','age']

让我们把每个特征与价格作图，以查看数据集及其特征。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000's)")
plt.show()

把每个特征与目标（价格）作图，可以大致看出哪些特征对价格影响最强。上面，面积增大价格也上升。卧室数和楼层数对价格似乎没有强影响。较新的住房价格高于较旧的住房。

<a name="toc_15456_5"></a>
## 多变量梯度下降
下面是你在上一个实验里推导出的多变量梯度下降公式：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j = 0..n-1}\newline
&b\ \ := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

其中，n 是特征数量，参数 $w_j$、$b$ 同时更新，且

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3}
\end{align}
$$
* m 是数据集中训练样本的数量

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是目标值


## 学习率
<figure>
    <img src="./图片/C1_W2_Lab06_learningrate.PNG" style="width:1200px;" >

> **图中文字（中文）**：梯度下降：b = b − α·∂/∂b J(w,b) “调试”：如何确保梯度下降工作正常；如何选择学习率 确保梯度下降正常工作：绘制迭代次数与代价的关系图（例如 100、200、300、400 次迭代） 如果 J(w,b) 反而增大，说明学习率过大（例如 α = 0.01 过大） 用梯度下降确定合适的学习率：先用较小的学习率，再逐步尝试更大的值
</figure>
课上讨论了设置学习率 $\\alpha$ 相关的一些问题。学习率控制参数更新的幅度，见上面的公式 (1)。它由所有参数共享。

让我们运行梯度下降，并在我们的数据集上尝试几个不同的 $\\alpha$ 取值

### $\alpha$ = 9.9e-7

In [ ]:
#把 alpha 设为 9.9e-7
_, _, hist = run_gradient_descent(X_train, y_train, 10, alpha = 9.9e-7)

看起来学习率过高。解没有收敛。代价不降反*升*。让我们把结果画出来：

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

右图显示了其中一个参数 $w_0$ 的取值。每次迭代它都越过最优值，结果代价最终不是趋近最小值，而是*上升*。注意这并非完整准确的图景，因为每一轮被修改的是 4 个参数而不是 1 个。这张图只展示了 $w_0$，其他参数固定在合适的取值上。在这张图以及后面的图中，你可能注意到蓝线和橙线略有偏差。


### $\alpha$ = 9e-7
让我们试一个稍小一点的值，看看会怎样。

In [ ]:
#把 alpha 设为 9e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 9e-7)

整个运行过程中代价都在下降，说明 alpha 不算过大。

In [ ]:
plot_cost_i_w(X_train, y_train, hist)

左图可见代价如预期般下降。右图可见 $w_0$ 仍在最小值附近振荡，但每次迭代是在减小而不是增大。注意上面 `dj_dw[0]` 每次都改变符号，因为 `w[0]` 越过了最优值。
这个 alpha 取值会收敛。你可以改变迭代次数，观察它的表现。

### $\alpha$ = 1e-7
让我们给 $\alpha$ 试一个稍小一点的值，看看会怎样。

In [ ]:
#把 alpha 设为 1e-7
_,_,hist = run_gradient_descent(X_train, y_train, 10, alpha = 1e-7)

整个运行过程中代价都在下降，说明 $\alpha$ 不算过大。 

In [ ]:
plot_cost_i_w(X_train,y_train,hist)

左图可见代价如预期般下降。右图可见 $w_0$ 一直在减小，没有越过最小值。注意上面整个运行过程中 `dj_w0` 都是负的。这个解也会收敛，只是不如上一个例子那么快。

## 特征缩放
<figure>
    <img src="./图片/C1_W2_Lab06_featurescalingheader.PNG" style="width:1200px;" >

> **图中文字（中文）**：特征缩放：让每个特征的取值范围大致落在 −1 到 1 之间 特征值与参数值：0 ≤ x_1 ≤ 2000，0.1 ≤ x_2 ≤ 5 特征缩放：目标是让每个特征大致处于 −1 ≤ x_j ≤ 1 若取值范围过大或过小，就做缩放 例如把 100 ≤ x ≤ 100 缩放到 −1 ≤ x ≤ 1
</figure>
课上介绍了重新缩放数据集以使各特征具有相近取值范围的重要性。
如果你对这样做的原因感兴趣，请点击下面的“详细信息”标题。如果不感兴趣，下面的小节将带你完成一次特征缩放的实现。

<details>
<summary>
    <font size='3', color='darkgreen'><b>详细信息</b></font>
</summary>

让我们再来看一下 $\\alpha$ = 9e-7 的情况。这非常接近在不发散的前提下我们能设置的最大 $\\alpha$ 值。下面是一次很短的运行，展示了最初几次迭代：

<figure>
    <img src="./图片/C1_W2_Lab06_ShortRun.PNG" style="width:1200px;" >

> **图中文字（中文）**：短程运行（Short run）的输出截图： 设置 alpha = 9e-7，运行 run_gradient_descent(x_train, y_train, …) - 代价从约 6.64616e+04 降到约 3.52385e+04 - 参数：w ≈ [1.73775314e-02, 8.27287625e-07, …]，b ≈ -9.3142283377918e-03 - w0 的梯度显著大于其他参数的梯度 结论：代价在下降；w0 正朝其最小值推进。
</figure>

上图中，虽然代价在下降，但很明显 $w_0$ 的推进速度比其他参数快得多，因为它的梯度大得多。

下面的图展示了 $\\alpha$ = 9e-7 时一次非常长的运行结果。这需要好几个小时。

<figure>
    <img src="./图片/C1_W2_Lab06_LongRun.PNG" style="width:1200px;" >

> **图中文字（中文）**：长时间运行（Long run）的输出截图： 设置 alpha = 9e-7，运行 run_gradient_descent(x_train, y_train, …) - 代价从约 6.46e+04 逐步下降到约 2.19e+02 - 参数：w0 ≈ 2.7e-01，w1 ≈ 2.9e+01，w2 ≈ 6.6e+01，w3 ≈ 1.4e+00，b ≈ 2.2e+00 - 梯度：dj_dw0 从 1.2e-03 降到 1.8e-05；dj_dw1 从 -1.0e+03 降到 -4.6e-02 等 结论：w0 很快达到接近最终值；w1、w2、w3 和 b 更新得慢得多。
</figure>
    
上图中可以看到，代价在最初下降之后下降得很慢。注意 `w0` 与 `w0`、`w1`、`w2` 之间的差别，以及 `dj_dw0` 与 `dj_dw1-3` 之间的差别。`w0` 很快达到接近最终值，且 `dj_dw0` 迅速下降到很小的值，说明 `w0` 已接近最终值。其他参数则下降得慢得多。

为什么会这样？有什么可以改进的地方吗？见下文：
<figure>
    <center> <img src="./图片/C1_W2_Lab06_scale.PNG"   >

> **图中文字（中文）**：repeat until convergence {     w_j = w_j − α·(1/m)Σ(…) } 其中公共误差项对所有参数都相同，而乘上的特征对每个参数都不同。 特征：面积（平方英尺）、卧室数、楼层数、房龄 在这个数据集中，该特征（面积）比其他特征大 2—3 个数量级。</center>
</figure>   

上图说明了为什么 $w$ 的更新不均匀。
- $\\alpha$ 由所有参数更新共享（$w$ 和 $b$）。
- 公共误差项会与 $w$ 对应的特征相乘（不与 $b$ 相乘）。
- 各特征的量级差别很大，使得有些特征的更新比其他特征快得多。在这个例子中，$w_0$ 乘以“面积（平方英尺）”，通常 > 1000；而 $w_1$ 乘以“卧室数量”，通常为 2-4。

解决办法就是特征缩放。

讲课中讨论了三种不同的技术：
- 特征缩放，本质上是把每个特征除以一个用户选定的值，使取值范围落在 -1 到 1 之间。
- 均值归一化：$x_i := \dfrac{x_i - \mu_i}{max - min} $ 
- z 分数归一化，我们将在下面探讨。


### z 分数归一化
经过 z 分数归一化后，所有特征的均值为 0、标准差为 1。

要实现 z 分数归一化，按下式调整你的输入值：
$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{4}$$ 
其中 $j$ 选择 X 矩阵中的一个特征或一列。$µ_j$ 是特征 (j) 所有取值的均值，$\sigma_j$ 是特征 (j) 的标准差。
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{5}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{6}
\end{align}
$$

>**实现注意事项：** 归一化特征时，务必要把用于归一化的值——计算中使用的均值和标准差——保存下来。在从模型学到参数之后，我们常常想预测此前未见过的住房价格。给定一个新的 x 值（客厅面积和卧室数），我们必须先用此前从训练集计算得到的均值和标准差对 x 做归一化。

**实现**

In [ ]:
def zscore_normalize_features(X):
    """计算按列做 z 分数归一化后的 X
   
    参数：
      X (ndarray): 形状 (m,n)，输入数据，m 个样本，n 个特征
     
    返回：
      X_norm (ndarray): 形状 (m,n)，按列归一化后的输入
      mu (ndarray):     形状 (n,)，每个特征的均值
      sigma (ndarray):  形状 (n,)，每个特征的标准差"""
    # 求每一列/每个特征的均值
    mu     = np.mean(X, axis=0)                 # mu 的形状为 (n,)
    # 求每一列/每个特征的标准差
    sigma  = np.std(X, axis=0)                  # sigma 的形状为 (n,)
    # 逐元素操作：从每个样本的该列中减去该列的均值，再除以该列的标准差
    X_norm = (X - mu) / sigma      

    return (X_norm, mu, sigma)
 
#检查一下我们的结果
#from sklearn.preprocessing import scale
#scale(X_orig, axis=0, with_mean=True, with_std=True, copy=True)

让我们看看 z 分数归一化涉及的步骤。下图逐步展示了这一变换过程。

In [ ]:
mu     = np.mean(X_train,axis=0)   
sigma  = np.std(X_train,axis=0) 
X_mean = (X_train - mu)
X_norm = (X_train - mu)/sigma      

fig,ax=plt.subplots(1, 3, figsize=(12, 3))
ax[0].scatter(X_train[:,0], X_train[:,3])
ax[0].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[0].set_title("unnormalized")
ax[0].axis('equal')

ax[1].scatter(X_mean[:,0], X_mean[:,3])
ax[1].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[1].set_title(r"X - $\mu$")
ax[1].axis('equal')

ax[2].scatter(X_norm[:,0], X_norm[:,3])
ax[2].set_xlabel(X_features[0]); ax[0].set_ylabel(X_features[3]);
ax[2].set_title(r"Z-score normalized")
ax[2].axis('equal')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle("distribution of features before, during, after normalization")
plt.show()

上图展示了训练集中两个参数“房龄”和“平方英尺”之间的关系。*它们用相同的比例绘制*。
- 左：未归一化：'面积（平方英尺）'特征的取值范围或方差远大于房龄
- 中：第一步从每个特征中减去均值或平均值。这样特征就以零为中心。'房龄'特征的变化不太看得出来，但'面积（平方英尺）'明显围绕零。
- 右：第二步除以方差。这样两个特征都以零为中心，且尺度相近。

让我们对数据做归一化，并与原始数据比较。

In [ ]:
# 对原始特征做归一化
X_norm, X_mu, X_sigma = zscore_normalize_features(X_train)
print(f"X_mu = {X_mu}, \nX_sigma = {X_sigma}")
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X_train,axis=0)}")   
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X_norm,axis=0)}")

归一化后，每一列的峰谷范围从数千倍缩小到 2-3 倍。

In [ ]:
fig,ax=plt.subplots(1, 4, figsize=(12, 3))
for i in range(len(ax)):
    norm_plot(ax[i],X_train[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count");
fig.suptitle("distribution of features before normalization")
plt.show()
fig,ax=plt.subplots(1,4,figsize=(12,3))
for i in range(len(ax)):
    norm_plot(ax[i],X_norm[:,i],)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("count"); 
fig.suptitle(f"distribution of features after normalization")

plt.show()

注意，上面归一化后数据的范围以零为中心、大致在 +/- 1 之间。最重要的是，各特征的取值范围相近。

让我们用归一化后的数据重新运行梯度下降算法。
注意 **alpha 的取值大了非常多**。这会加速下降。

In [ ]:
w_norm, b_norm, hist = run_gradient_descent(X_norm, y_train, 1000, 1.0e-1, )

缩放后的特征**快得多得多地**得到了非常准确的结果！注意在这段相当短的运行结束时，每个参数的梯度都很小。对于归一化特征后的回归，学习率 0.1 是个不错的起点。
让我们把预测值与目标值作图。注意，预测是用归一化后的特征做出的，而图上显示的是原始特征值。

In [ ]:
#用归一化后的特征预测目标
m = X_norm.shape[0]
yp = np.zeros(m)
for i in range(m):
    yp[i] = np.dot(X_norm[i], w_norm) + b_norm

    # 绘制预测值与目标值随原始特征的变化
fig,ax=plt.subplots(1,4,figsize=(12, 3),sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:,i],y_train, label = 'target')
    ax[i].set_xlabel(X_features[i])
    ax[i].scatter(X_train[:,i],yp,color=dlorange, label = 'predict')
ax[0].set_ylabel("Price"); ax[0].legend();
fig.suptitle("target versus prediction using z-score normalized model")
plt.show()

结果看起来不错。有几点需要注意：
- 有多个特征时，我们无法再用一张图同时展示结果与各特征的关系。
- 生成图时使用的是归一化后的特征。任何使用从归一化训练集学到的参数所做的预测，也必须做归一化。

**预测**
我们生成模型的目的，是用它来预测不在数据集中的住房价格。让我们预测一套 1200 平方英尺、3 间卧室、1 层、房龄 40 年的住房的价格。记住，你必须用在归一化训练数据时得到的均值和标准差对数据做归一化。 

In [ ]:
# 首先，对样本做归一化。
x_house = np.array([1200, 3, 1, 40])
x_house_norm = (x_house - X_mu) / X_sigma
print(x_house_norm)
x_house_predict = np.dot(x_house_norm, w_norm) + b_norm
print(f" predicted price of a house with 1200 sqft, 3 bedrooms, 1 floor, 40 years old = ${x_house_predict*1000:0.0f}")

**代价等高线**  
<img align="left" src="./图片/C1_W2_Lab06_contours.PNG"   style="width:240px;" >

> **图中文字（中文）**：左：未重新缩放的“卧室数”与“面积（平方英尺）”的代价等高线 右：重新缩放后的代价等高线另一种观察特征缩放的角度是看代价等高线。当特征尺度不匹配时，等高线图中代价随参数变化的图形是不对称的。

下图中参数的尺度是匹配的。左图是归一化特征之前，代价关于 w[0]（平方英尺）与 w[1]（卧室数量）的等高线图。该图非常不对称，以至于构成等高线的曲线都看不出来。相反，当特征被归一化后，代价等高线要对称得多。结果是梯度下降过程中各参数的更新能够取得大致相同的进展。


In [ ]:
plt_equal_scale(X_train, X_norm, y_train)


## 恭喜！
在本实验中你：
- 使用了你在之前实验中开发的、支持多个特征的线性回归例程
- 探讨了学习率 $\alpha$ 对收敛的影响
- 发现了使用 z 分数归一化进行特征缩放在加速收敛方面的价值

## 致谢
本住房数据源自 Dean De Cock 为数据科学教育编制整理的 [Ames 住房数据集](http://jse.amstat.org/v19n3/decock.pdf)。